# 10 — WMH volume and infarct-volume-stratified analyses

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import nibabel as nib
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import load_label, restricted_pool, build_demographics_matrix
from analysis.lib.pipelines import (process_fold, process_fold_majority,
                                    process_fold_permuted)
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA     = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA"
DERIV      = LACUNA / "derivatives"
WMH_DIR    = PROJECT_ROOT / "outputs" / "predictive_modeling_wmh"
VT_DIR     = WMH_DIR / "volume_vs_topology"
STRATA_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_volume_strata"
for d in (WMH_DIR, VT_DIR, STRATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

TARGETS = [
    "cognition-PSCI_present",
    "cognition-aef_domain_score",
    "cognition-ps_domain_score",
    "cognition-language_domain_score",
    "cognition-verbalmemory_domain_score",
    "cognition-vsfunctions_domain_score",
    "cognition-vsmemory_domain_score",
]

clinical_full = pd.read_csv(CLIN_PATH, index_col=0)

In [ ]:
# 1. Per-subject WMH volumes from binary masks
def _wmh_mask_path(sid):
    return (LACUNA / f"sub-{sid}" / "ses-01" / "anat" /
            f"sub-{sid}_ses-01_label-WMH_space-MNI152NLin6Asym_mask.nii.gz")


def volume_one(sid):
    p = _wmh_mask_path(sid)
    if not p.exists():
        return {"subject_id": sid, "wmh_volume_ml": np.nan,
                "n_voxels": 0, "voxel_mm3": np.nan}
    img = nib.load(p)
    arr = np.asarray(img.dataobj, dtype=np.float32)
    n_vox = int((arr > 0).sum())
    vox_mm3 = float(abs(np.linalg.det(img.affine[:3, :3])))
    return {"subject_id": sid, "wmh_volume_ml": n_vox * vox_mm3 / 1000.0,
            "n_voxels": n_vox, "voxel_mm3": vox_mm3}


VOL_CSV = WMH_DIR / "wmh_volumes.csv"
if VOL_CSV.exists():
    print(f"[skip] {VOL_CSV} exists")
else:
    pool = restricted_pool(DERIV, clinical_full, ["WMH"])
    print(f"computing WMH volume for {len(pool)} subjects")
    rows = Parallel(n_jobs=-1)(delayed(volume_one)(sid) for sid in pool)
    df = pd.DataFrame(rows)
    print(f"  {int(df['wmh_volume_ml'].notna().sum())}/{len(df)} masks found")
    df.to_csv(VOL_CSV, index=False)
    print(f"saved {VOL_CSV}")

In [ ]:
# 2. WMH volume-only vs topography+connectivity (imaging-only stacker)
if (VT_DIR / "stacking_scores.csv").exists():
    print(f"[skip] volume-vs-topology outputs exist")
else:
    pool = restricted_pool(DERIV, clinical_full, ["WMH"])
    clinical = clinical_full.loc[pool].copy()
    print(f"WMH pool: N = {len(pool)}")

    wmh_lesion = load_label(DERIV, "WMH", "rd",  "damagepct").reindex(pool)
    wmh_flnm   = load_label(DERIV, "WMH", "fnm", "tmap").reindex(pool)
    wmh_slnm   = load_label(DERIV, "WMH", "snm", "disconnectionpct").reindex(pool)

    vol = pd.read_csv(VOL_CSV).set_index("subject_id")
    clinical["wmh_volume_ml"] = vol.reindex(pool)["wmh_volume_ml"].values

    X_vol_only = clinical[["wmh_volume_ml"]].values
    n_demo_vol = 0
    cv = pooled_cv(seed=42)
    all_scores, stacker_scores = [], []

    for target in TARGETS:
        y_series = clinical[target]
        valid = y_series.notna()
        if valid.sum() < 50:
            continue
        y = y_series[valid].values.astype(int)
        if y.sum() < 10 or len(y) - y.sum() < 10:
            continue
        Xv = X_vol_only[valid.values]
        print(f"--- {target}: N={len(y)} ---")
        splits = list(cv.split(Xv, y))

        for fold_id, (tr, te) in enumerate(splits):
            r = process_fold_majority(fold_id, tr, te, y, target)
            r["Modality"] = "base"; all_scores.append(r)
            r = process_fold_permuted(fold_id, tr, te, Xv, y, target, n_demo_vol,
                                      rng_seed=2024)
            r["Modality"] = "base"; all_scores.append(r)

        fr = Parallel(n_jobs=-1)(
            delayed(process_fold)(i, tr, te, Xv, y, "Volume", target, n_demo_vol)
            for i, (tr, te) in enumerate(splits))
        for r in fr:
            r["Modality"] = "wmh_volume"
        all_scores.extend(fr)
        print(f"  Volume   AUPRC = {np.nanmean([r['AUPRC'] for r in fr]):.3f}")

        view_configs = [
            ("lesion_wmh", wmh_lesion[valid.values].values),
            ("flnm_wmh",   wmh_flnm[valid.values].values),
            ("slnm_wmh",   wmh_slnm[valid.values].values),
        ]
        sr = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, tr, te, view_configs, y, 0)
            for i, (tr, te) in enumerate(splits))
        for r in sr:
            r["Target"] = target; r["Model"] = "Topology"; r["Modality"] = "meta"
        stacker_scores.extend(sr)
        print(f"  Topology AUPRC = {np.nanmean([r['AUPRC'] for r in sr]):.3f}")

    pd.DataFrame(all_scores).to_csv(VT_DIR / "all_scores.csv", index=False)
    pd.DataFrame(stacker_scores).to_csv(VT_DIR / "stacking_scores.csv", index=False)
    df = pd.concat([pd.DataFrame(all_scores), pd.DataFrame(stacker_scores)],
                   ignore_index=True)
    summ = df.groupby(["Target", "Model", "Modality"])["AUPRC"].agg(
        ["mean", "std", "count"]).reset_index()
    summ["sem"] = summ["std"] / np.sqrt(summ["count"])
    summ.to_csv(VT_DIR / "summary.csv", index=False)
    print(f"saved {VT_DIR}")

In [ ]:
# 3. Infarct-volume-stratified core models
STRATA = [
    ("small",  lambda v: v < 5,               "< 5 mL  (lacunar)"),
    ("medium", lambda v: (v >= 5) & (v < 30), "5–30 mL  (moderate)"),
    ("large",  lambda v: v >= 30,             "≥ 30 mL  (large territorial)"),
]


def run_stratum(stratum_name, stratum_label, pool_idx, clinical,
                lesion_acute, flnm_acute, slnm_acute,
                drop_cols=("stroke-total_infarct_volume_ml",)):
    print(f"=== stratum '{stratum_name}' {stratum_label} (N={len(pool_idx)}) ===")
    clin_s = clinical.loc[pool_idx]
    X_demo, demo_names, _, _, _ = build_demographics_matrix(clin_s, exclude_cohort=False)
    # Drop the stratification variable (constant-ish within a stratum)
    drop_idx = [i for i, n in enumerate(demo_names) if n in drop_cols]
    if drop_idx:
        keep = [i for i in range(len(demo_names)) if i not in drop_idx]
        X_base_pool = X_demo[:, keep]
    else:
        X_base_pool = X_demo
    n_demo = X_base_pool.shape[1]

    feat = {
        "lesion_acute": lesion_acute.loc[pool_idx].values,
        "flnm_acute":   flnm_acute.loc[pool_idx].values,
        "slnm_acute":   slnm_acute.loc[pool_idx].values,
    }
    cv = pooled_cv(seed=42)

    all_scores, stacker_scores = [], []
    for target in TARGETS:
        y_series = clin_s[target]
        valid = y_series.notna()
        y = y_series[valid].values.astype(int)
        if y.sum() < 10 or len(y) - y.sum() < 10:
            print(f"  [skip] {target}")
            continue
        X_base = X_base_pool[valid.values]
        splits = list(cv.split(X_base, y))

        for fold_id, (tr, te) in enumerate(splits):
            r = process_fold_majority(fold_id, tr, te, y, target)
            r["Modality"] = "base"; r["Stratum"] = stratum_name; all_scores.append(r)
            r = process_fold_permuted(fold_id, tr, te, X_base, y, target, n_demo,
                                      rng_seed=2024)
            r["Modality"] = "base"; r["Stratum"] = stratum_name; all_scores.append(r)

        configs = {
            ("Baseline", "base"):   X_base,
            ("Acute",    "Lesion"): np.hstack([X_base, feat["lesion_acute"][valid.values]]),
            ("Acute",    "fLNM"):   np.hstack([X_base, feat["flnm_acute"][valid.values]]),
            ("Acute",    "sLNM"):   np.hstack([X_base, feat["slnm_acute"][valid.values]]),
        }
        for (model_name, modality), X in configs.items():
            fold_results = Parallel(n_jobs=-1)(
                delayed(process_fold)(i, tr, te, X, y, model_name, target, n_demo)
                for i, (tr, te) in enumerate(splits))
            for r in fold_results:
                r["Modality"] = modality
                r["Stratum"] = stratum_name
            all_scores.extend(fold_results)

        view_configs = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, feat["lesion_acute"][valid.values]])),
            ("flnm_acute",   np.hstack([X_base, feat["flnm_acute"][valid.values]])),
            ("slnm_acute",   np.hstack([X_base, feat["slnm_acute"][valid.values]])),
        ]
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, tr, te, view_configs, y, n_demo)
            for i, (tr, te) in enumerate(splits))
        for r in fold_results:
            r["Target"] = target; r["Model"] = "Stacking"
            r["Modality"] = "meta"; r["Stratum"] = stratum_name
        stacker_scores.extend(fold_results)
        print(f"  {target}: stacker AUPRC = "
              f"{np.nanmean([r['AUPRC'] for r in fold_results]):.3f}")

    return all_scores, stacker_scores


def run_stratification(out_dir, strata, drop_cols=()):
    """Run the per-stratum model battery; strata = [(name, mask_fn, label)]."""
    if (out_dir / "stacking_scores.csv").exists():
        print(f"[skip] {out_dir.name} outputs exist")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    acute_pool = restricted_pool(DERIV, clinical_full, ["acuteinfarct"])
    lesion_acute = load_label(DERIV, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
    flnm_acute   = load_label(DERIV, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
    slnm_acute   = load_label(DERIV, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)

    all_rows, stacker_rows = [], []
    for stratum_name, mask_fn, label in strata:
        mask = mask_fn(clinical_full.loc[acute_pool])
        pool_idx = clinical_full.loc[acute_pool].index[mask.values]
        if len(pool_idx) < 50:
            print(f"[skip stratum {stratum_name}: N={len(pool_idx)}]")
            continue
        a, s = run_stratum(stratum_name, label, pool_idx, clinical_full,
                           lesion_acute, flnm_acute, slnm_acute,
                           drop_cols=drop_cols)
        all_rows.extend(a)
        stacker_rows.extend(s)

    pd.DataFrame(all_rows).to_csv(out_dir / "all_scores.csv", index=False)
    pd.DataFrame(stacker_rows).to_csv(out_dir / "stacking_scores.csv", index=False)
    df = pd.concat([pd.DataFrame(all_rows), pd.DataFrame(stacker_rows)], ignore_index=True)
    summ = df.groupby(["Stratum", "Target", "Model", "Modality"])["AUPRC"].agg(
        ["mean", "std", "count"]).reset_index()
    summ["sem"] = summ["std"] / np.sqrt(summ["count"])
    summ.to_csv(out_dir / "summary.csv", index=False)
    print(f"saved {out_dir}")


def _vol(df):
    return pd.to_numeric(df["stroke-total_infarct_volume_ml"], errors="coerce")


run_stratification(STRATA_DIR, [
    (name, (lambda df, p=pred: _vol(df).notna() & pd.Series(p(_vol(df).values),
                                                            index=df.index)), label)
    for name, pred, label in STRATA
], drop_cols=("stroke-total_infarct_volume_ml",))

In [ ]:
# Age- and sex-stratified pooled CV (supplement). Age tertiles computed on the
# acute pool; the stratification variable is dropped from the baseline within
# its own strata.
AGE_COL, SEX_COL = "demographics-age", "demographics-sex"
AGE_STRATA_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_age_strata"
SEX_STRATA_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_sex_strata"

_pool_for_cuts = restricted_pool(DERIV, clinical_full, ["acuteinfarct"])
_age = pd.to_numeric(clinical_full.loc[_pool_for_cuts, AGE_COL], errors="coerce")
_a1, _a2 = _age.quantile([1 / 3, 2 / 3])
print(f"age tertile cuts: {_a1:.1f} / {_a2:.1f}")


def _age_mask(lo, hi):
    def fn(df):
        a = pd.to_numeric(df[AGE_COL], errors="coerce")
        return a.notna() & (a >= lo) & (a < hi)
    return fn


run_stratification(AGE_STRATA_DIR, [
    ("young", _age_mask(-np.inf, _a1), f"age < {_a1:.0f}"),
    ("middle", _age_mask(_a1, _a2),    f"{_a1:.0f} <= age < {_a2:.0f}"),
    ("old",   _age_mask(_a2, np.inf),  f"age >= {_a2:.0f}"),
], drop_cols=(AGE_COL,))


def _sex_mask(val):
    def fn(df):
        s = pd.to_numeric(df[SEX_COL], errors="coerce")
        return s.notna() & (s == val)
    return fn


_sex_vals = sorted(pd.to_numeric(clinical_full[SEX_COL], errors="coerce")
                   .dropna().unique())
print(f"sex coding values: {_sex_vals}")
run_stratification(SEX_STRATA_DIR, [
    (f"sex_{int(v)}", _sex_mask(v), f"{SEX_COL} == {int(v)}") for v in _sex_vals
], drop_cols=(SEX_COL,))